# 예제3: 사전학습 모델(Helsinki-NLP)을 이용한 실제 번역
## 목표: 실제 번역 모델로 번역 수행 및 BLEU 평가

### 학습 내용
1. 사전학습 모델 로드
2. 실제 번역 수행
3. BLEU 점수로 평가
4. 번역 품질 분석

## STEP 1: 라이브러리 설치

In [ ]:
# 필수 라이브러리
!pip install transformers torch -q
!pip install sacrebleu -q

from transformers import MarianMTModel, MarianTokenizer
import torch
from sacrebleu.metrics import BLEU
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")
print("✅ 라이브러리 설치 및 임포트 완료")

## STEP 2: 번역 모델 로드

In [ ]:
print("모델 로드 중... (첫 실행 시 다운로드 포함)")

# 영어 → 한국어 모델
model_name = 'Helsinki-NLP/Opus-MT-en-ko'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

print(f"✅ 모델 로드 완료: {model_name}")
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")

## STEP 3: 번역 함수 정의

In [ ]:
def translate_text(text, model, tokenizer, device):
    """
    텍스트를 영어에서 한국어로 번역
    
    Args:
        text: 번역할 영어 텍스트
        model: 번역 모델
        tokenizer: 토크나이저
        device: 실행 디바이스
    
    Returns:
        번역된 한국어 텍스트
    """
    
    # 입력 토큰화
    inputs = tokenizer(text, return_tensors='pt', padding=True).to(device)
    
    # 번역
    with torch.no_grad():
        translated = model.generate(**inputs)
    
    # 디코딩
    output_text = tokenizer.batch_decode(translated, skip_special_tokens=True)
    
    return output_text[0]

print("✅ 번역 함수 정의 완료")

## STEP 4: 번역 테스트

In [ ]:
# 테스트 문장
test_sentences = [
    'Hello, how are you?',
    'I love natural language processing.',
    'Machine translation is important for global communication.',
    'This is a test sentence for translation.',
    'The weather is beautiful today.',
]

print("\n【번역 테스트】\n")

translations = []

for eng_text in test_sentences:
    kor_text = translate_text(eng_text, model, tokenizer, device)
    translations.append((eng_text, kor_text))
    print(f"영어: {eng_text}")
    print(f"한국어: {kor_text}")
    print()

## STEP 5: BLEU 평가

In [ ]:
# 참조 번역 (수동으로 번역)
references = [
    ['안녕하세요 어떻게 지내세요'],  # "Hello, how are you?"
    ['나는 자연어 처리를 좋아합니다'],  # "I love natural language processing."
    ['기계 번역은 세계적 소통에 중요합니다'],  # "Machine translation is important..."
    ['이것은 번역을 위한 테스트 문장입니다'],  # "This is a test sentence..."
    ['오늘 날씨가 정말 좋습니다'],  # "The weather is beautiful today."
]

# 생성된 번역 (모델의 출력)
hypotheses = [trans[1] for trans in translations]

# BLEU 점수 계산
bleu = BLEU()
score = bleu.corpus_score(hypotheses, [references])

print("\n【BLEU 평가 결과】\n")
print(f"BLEU 점수: {score.score:.2f}")
print(f"\n세부 점수:")
print(f"  1-gram (Unigram): {score.precisions[0]:.2f}")
print(f"  2-gram (Bigram): {score.precisions[1]:.2f}")
print(f"  3-gram (Trigram): {score.precisions[2]:.2f}")
print(f"  4-gram (4-gram): {score.precisions[3]:.2f}")
print(f"\n길이 페널티 (BP): {score.bp:.3f}")
print(f"참조 길이: {score.ref_len}")
print(f"생성 길이: {score.sys_len}")

## STEP 6: 번역 품질 상세 분석

In [ ]:
import pandas as pd
from collections import Counter

def simple_bleu(reference, hypothesis, max_n=4):
    """간단한 BLEU 계산"""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    
    precisions = []
    for n in range(1, max_n + 1):
        if len(hyp_tokens) < n:
            precisions.append(0.0)
            continue
        
        ref_ngrams = Counter([' '.join(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1)])
        hyp_ngrams = Counter([' '.join(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens)-n+1)])
        
        matching = 0
        for ngram in hyp_ngrams:
            matching += min(hyp_ngrams[ngram], ref_ngrams.get(ngram, 0))
        
        precision = matching / sum(hyp_ngrams.values()) if sum(hyp_ngrams.values()) > 0 else 0
        precisions.append(precision)
    
    return sum(precisions) / len(precisions)

# 각 문장별 BLEU 계산
print("\n【문장별 BLEU 점수】\n")

results = []
for i, ((eng, kor), ref_list) in enumerate(zip(translations, references), 1):
    ref = ref_list[0]
    bleu_score = simple_bleu(ref, kor)
    
    results.append({
        '번호': i,
        '영어': eng[:30],
        '참조': ref[:20],
        '번역': kor[:20],
        'BLEU': f"{bleu_score:.3f}"
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\n평균 BLEU: {df['BLEU'].astype(float).mean():.3f}")

## STEP 7: 번역 길이 분석

In [ ]:
print("\n【번역 길이 분석】\n")

for i, ((eng, kor), ref_list) in enumerate(zip(translations, references), 1):
    ref = ref_list[0]
    
    eng_len = len(eng.split())
    ref_len = len(ref.split())
    kor_len = len(kor.split())
    
    length_ratio = kor_len / ref_len if ref_len > 0 else 0
    
    print(f"문장 {i}:")
    print(f"  원문 길이: {eng_len}")
    print(f"  참조 길이: {ref_len}")
    print(f"  번역 길이: {kor_len}")
    print(f"  길이 비율: {length_ratio:.2f}")
    print()

## STEP 8: 다양한 언어 쌍 시험 (선택)

In [ ]:
print("\n【지원하는 언어 쌍 예시】")
print("""
Helsinki-NLP에서 제공하는 번역 모델:
  - en-ko: 영어 → 한국어
  - en-ja: 영어 → 일본어
  - en-zh: 영어 → 중국어
  - en-es: 영어 → 스페인어
  - en-fr: 영어 → 프랑스어
  - en-de: 영어 → 독일어
  - en-ru: 영어 → 러시아어
  등 100+ 언어 쌍

사용 방법:
  model_name = 'Helsinki-NLP/Opus-MT-en-{target_lang}'
  tokenizer = MarianTokenizer.from_pretrained(model_name)
  model = MarianMTModel.from_pretrained(model_name)
""")

## 🎯 정리
- 사전학습 모델을 이용한 실제 번역 수행
- BLEU를 이용한 번역 품질 평가
- 다양한 언어 쌍 지원